## 代谢组/脂质组表型整理

这一步把原始表型文件整理成一个干净、适合下游分析的表。

主要处理内容：

1. 读取原始代谢组/脂质组结果表  
2. 清洗 `sample` 列，统一样本名格式  
   - 例如 `CIMA_H019` -> `CIMA19`
   - `CIMA_H015\t` -> `CIMA15`
   - `CIMA_451` -> `CIMA451`
3. 其余列尽量转成数值型
4. 输出整理后的主表
5. 额外输出一个 sample 映射表，便于人工核对


In [1]:
from pathlib import Path
import re
import pandas as pd

# ===== 路径设置（相对项目根目录）=====
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and PROJECT_ROOT.name != "CIMA_multiomics_regulation":
    PROJECT_ROOT = PROJECT_ROOT.parent

if PROJECT_ROOT.name != "CIMA_multiomics_regulation":
    raise RuntimeError("未找到项目根目录 CIMA_multiomics_regulation，请在项目目录内运行该脚本。")

INPUT_FILE = PROJECT_ROOT / "data/raw/CIMA/Metabolites_and_Lipids/CIMA_Sample_Plasma_Metabolites_and_Lipids_Results.csv"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/CIMA/phenotype"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "CIMA_metabolites_lipids_clean.tsv"
SAMPLE_MAP_FILE = OUTPUT_DIR / "CIMA_metabolites_lipids_sample_map.tsv"

print(f"INPUT_FILE = {INPUT_FILE}")
print(f"exists = {INPUT_FILE.exists()}")

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"找不到输入文件: {INPUT_FILE}")


# ===== 读取原始表 =====
df = pd.read_csv(INPUT_FILE)
print("原始表 shape:", df.shape)
print("前5列:", df.columns[:5].tolist())

if "sample" not in df.columns:
    raise ValueError("输入表中没有找到 'sample' 列，请先检查原始文件列名。")


# ===== sample 清洗函数 =====
def clean_sample_id(x):
    """
    统一样本名格式：
    - 去掉首尾空格、制表符
    - CIMA_H019 -> CIMA19
    - CIMA_H015\t -> CIMA15
    - CIMA_451 -> CIMA451
    - 尽量保留 CIMA + 数字 的格式
    """
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()
    s = s.replace("\t", "").replace(" ", "")
    s = s.upper()

    m = re.match(r"^CIMA(?:[_-]?H|[_-])?0*([0-9]+)$", s)
    if m:
        return f"CIMA{int(m.group(1))}"

    m2 = re.search(r"CIMA.*?([0-9]+)$", s)
    if m2:
        return f"CIMA{int(m2.group(1))}"

    return s


# ===== 清洗 sample =====
sample_map = df[["sample"]].copy()
sample_map["sample_raw"] = sample_map["sample"]
sample_map["sample_clean"] = sample_map["sample_raw"].map(clean_sample_id)

df["sample"] = df["sample"].map(clean_sample_id)

other_cols = [c for c in df.columns if c != "sample"]
df = df[["sample"] + other_cols]


# ===== 数值列转 numeric =====
for col in df.columns:
    if col == "sample":
        continue
    df[col] = pd.to_numeric(df[col], errors="coerce")


# ===== 基本检查 =====
print("\n清洗后 shape:", df.shape)
print("sample 示例:")
print(df["sample"].head().tolist())

print("\n缺失 sample 数量:", df["sample"].isna().sum())
print("sample 去重后数量:", df["sample"].nunique(dropna=True))
print("是否有重复 sample:", df["sample"].duplicated().any())

dup_df = df[df["sample"].duplicated(keep=False)].sort_values("sample")
if len(dup_df) > 0:
    print("\n重复 sample 示例:")
    print(dup_df[["sample"]].head(20))


# ===== sample 映射表整理 =====
sample_map = sample_map.rename(columns={"sample": "sample_original"})
sample_map = sample_map[["sample_raw", "sample_clean"]].drop_duplicates()

print("\nsample 映射表示例:")
print(sample_map.head(10))


# ===== 保存结果 =====
df.to_csv(OUTPUT_FILE, sep="\t", index=False)
sample_map.to_csv(SAMPLE_MAP_FILE, sep="\t", index=False)

print("\n已保存：")
print(OUTPUT_FILE)
print(SAMPLE_MAP_FILE)


INPUT_FILE = /data/work/CIMA_multiomics_regulation/data/raw/CIMA/Metabolites_and_Lipids/CIMA_Sample_Plasma_Metabolites_and_Lipids_Results.csv
exists = True
原始表 shape: (390, 1550)
前5列: ['sample', 'O-Phosphoryl-ethanolamine', 'Acesulfame', 'Acetic acid', 'Propanoic acid']

清洗后 shape: (390, 1550)
sample 示例:
['CIMA15', 'CIMA16', 'CIMA17', 'CIMA18', 'CIMA19']

缺失 sample 数量: 0
sample 去重后数量: 390
是否有重复 sample: False

sample 映射表示例:
  sample_raw sample_clean
0  CIMA_H015       CIMA15
1  CIMA_H016       CIMA16
2  CIMA_H017       CIMA17
3  CIMA_H018       CIMA18
4  CIMA_H019       CIMA19
5  CIMA_H020       CIMA20
6  CIMA_H021       CIMA21
7  CIMA_H022       CIMA22
8  CIMA_H023       CIMA23
9  CIMA_H024       CIMA24

已保存：
/data/work/CIMA_multiomics_regulation/data/processed/CIMA/phenotype/CIMA_metabolites_lipids_clean.tsv
/data/work/CIMA_multiomics_regulation/data/processed/CIMA/phenotype/CIMA_metabolites_lipids_sample_map.tsv


: 

## 代谢组/脂质组表型表整理为 BGE_ID 版本

这一步把原始表型文件里的 `sample` 清洗后，对应到 meta 表中的 `BGE_ID`，最后输出一个以后续分析可直接使用的表型文件。

处理逻辑：

1. 读取原始代谢组/脂质组表  
2. 清洗 `sample`，统一成 `CIMA15`、`CIMA451` 这种格式  
3. 读取 `basic_id_sex_age.csv`，取其中的 `BGE_ID` 和 `CIMA_ID`  
4. 用 `sample_clean == CIMA_ID` 做匹配  
5. 主表最终只保留 `BGE_ID` 作为样本标识  
6. 另外输出一个映射表和未匹配样本表，方便检查

In [2]:
from pathlib import Path
import re
import pandas as pd

# ===== 路径设置（相对项目根目录）=====
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and PROJECT_ROOT.name != "CIMA_multiomics_regulation":
    PROJECT_ROOT = PROJECT_ROOT.parent

if PROJECT_ROOT.name != "CIMA_multiomics_regulation":
    raise RuntimeError("未找到项目根目录 CIMA_multiomics_regulation，请在项目目录内运行该脚本。")

INPUT_FILE = PROJECT_ROOT / "data/raw/CIMA/Metabolites_and_Lipids/CIMA_Sample_Plasma_Metabolites_and_Lipids_Results.csv"
META_FILE = PROJECT_ROOT / "data/processed/CIMA/meta_inventory/basic_id_sex_age.csv"

OUTPUT_DIR = PROJECT_ROOT / "data/processed/CIMA/phenotype"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "CIMA_metabolites_lipids_pheno_bge.tsv"
SAMPLE_MAP_FILE = OUTPUT_DIR / "CIMA_metabolites_lipids_sample_map.tsv"
UNMATCHED_FILE = OUTPUT_DIR / "CIMA_metabolites_lipids_unmatched_samples.tsv"

print(f"INPUT_FILE = {INPUT_FILE}")
print(f"META_FILE  = {META_FILE}")
print(f"INPUT exists = {INPUT_FILE.exists()}")
print(f"META  exists = {META_FILE.exists()}")

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"找不到输入文件: {INPUT_FILE}")
if not META_FILE.exists():
    raise FileNotFoundError(f"找不到 meta 文件: {META_FILE}")


# ===== 读取数据 =====
df = pd.read_csv(INPUT_FILE)
meta = pd.read_csv(META_FILE)

print("\n原始表 shape:", df.shape)
print("meta 表 shape:", meta.shape)

if "sample" not in df.columns:
    raise ValueError("输入表中没有找到 'sample' 列。")

required_meta_cols = ["BGE_ID", "CIMA_ID"]
missing_meta_cols = [c for c in required_meta_cols if c not in meta.columns]
if missing_meta_cols:
    raise ValueError(f"meta 表缺少必要列: {missing_meta_cols}")


# ===== sample 清洗函数 =====
def clean_sample_id(x):
    """
    统一样本名格式：
    - 去掉首尾空格、制表符
    - CIMA_H019 -> CIMA19
    - CIMA_H015\t -> CIMA15
    - CIMA_451 -> CIMA451
    """
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()
    s = s.replace("\t", "").replace(" ", "")
    s = s.upper()

    m = re.match(r"^CIMA(?:[_-]?H|[_-])?0*([0-9]+)$", s)
    if m:
        return f"CIMA{int(m.group(1))}"

    m2 = re.search(r"CIMA.*?([0-9]+)$", s)
    if m2:
        return f"CIMA{int(m2.group(1))}"

    return s


# ===== 清洗 sample，并准备 meta =====
df["sample_raw"] = df["sample"]
df["sample_clean"] = df["sample_raw"].map(clean_sample_id)

meta = meta.copy()
meta["CIMA_ID"] = meta["CIMA_ID"].astype(str).str.strip().str.upper()
meta["BGE_ID"] = meta["BGE_ID"].astype(str).str.strip()


# ===== sample 对照表 =====
sample_map = (
    df[["sample_raw", "sample_clean"]]
    .drop_duplicates()
    .merge(
        meta[["BGE_ID", "CIMA_ID"]],
        left_on="sample_clean",
        right_on="CIMA_ID",
        how="left"
    )
    [["sample_raw", "sample_clean", "BGE_ID"]]
    .drop_duplicates()
)

print("\nsample 对照表示例:")
print(sample_map.head(10))


# ===== 合并 BGE_ID 回主表 =====
df = df.merge(
    meta[["BGE_ID", "CIMA_ID"]],
    left_on="sample_clean",
    right_on="CIMA_ID",
    how="left"
)

# 未匹配样本
unmatched = (
    df.loc[df["BGE_ID"].isna(), ["sample_raw", "sample_clean"]]
    .drop_duplicates()
    .copy()
)

# 只保留匹配成功的样本
df = df[df["BGE_ID"].notna()].copy()

# 去掉不再需要的列
drop_cols = [c for c in ["sample", "sample_raw", "sample_clean", "CIMA_ID"] if c in df.columns]
df = df.drop(columns=drop_cols)

# 把 BGE_ID 放到最前
other_cols = [c for c in df.columns if c != "BGE_ID"]
df = df[["BGE_ID"] + other_cols]

# 其余列尽量转成数值
for col in df.columns:
    if col == "BGE_ID":
        continue
    df[col] = pd.to_numeric(df[col], errors="coerce")


# ===== 检查 =====
print("\n匹配统计:")
print(f"sample 去重后总数: {len(sample_map)}")
print(f"成功匹配 BGE_ID:   {sample_map['BGE_ID'].notna().sum()}")
print(f"未匹配样本数:      {sample_map['BGE_ID'].isna().sum()}")

print("\n主表 shape:", df.shape)
print("主表前几列:", df.columns[:10].tolist())
print("BGE_ID 是否重复:", df["BGE_ID"].duplicated().any())

if len(unmatched) > 0:
    print("\n未匹配样本示例:")
    print(unmatched.head(20))


# ===== 保存 =====
df.to_csv(OUTPUT_FILE, sep="\t", index=False)
sample_map.to_csv(SAMPLE_MAP_FILE, sep="\t", index=False)
unmatched.to_csv(UNMATCHED_FILE, sep="\t", index=False)

print("\n已保存：")
print(OUTPUT_FILE)
print(SAMPLE_MAP_FILE)
print(UNMATCHED_FILE)

INPUT_FILE = /data/work/CIMA_multiomics_regulation/data/raw/CIMA/Metabolites_and_Lipids/CIMA_Sample_Plasma_Metabolites_and_Lipids_Results.csv
META_FILE  = /data/work/CIMA_multiomics_regulation/data/processed/CIMA/meta_inventory/basic_id_sex_age.csv
INPUT exists = True
META  exists = True

原始表 shape: (390, 1550)
meta 表 shape: (467, 4)

sample 对照表示例:
  sample_raw sample_clean          BGE_ID
0  CIMA_H015       CIMA15  E-B21433880158
1  CIMA_H016       CIMA16  E-B21454468614
2  CIMA_H017       CIMA17  E-B21481266996
3  CIMA_H018       CIMA18  E-B21519886974
4  CIMA_H019       CIMA19  E-B21540551243
5  CIMA_H020       CIMA20  E-B21546282403
6  CIMA_H021       CIMA21  E-B21559280278
7  CIMA_H022       CIMA22  E-B21576997531
8  CIMA_H023       CIMA23  E-B21588672872
9  CIMA_H024       CIMA24  E-B21622568877

匹配统计:
sample 去重后总数: 390
成功匹配 BGE_ID:   390
未匹配样本数:      0

主表 shape: (390, 1550)
主表前几列: ['BGE_ID', 'O-Phosphoryl-ethanolamine', 'Acesulfame', 'Acetic acid', 'Propanoic acid', 'Glycolic a

: 